<a href="https://colab.research.google.com/github/Sashwit/IndependentWork/blob/main/XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# XGBoost Implementation for Customer Churn Prediction

1. Initialize with a base prediction
2. For each iteration, compute gradients and hessians
3. Build a tree to minimize the loss function
4. Update the model with the new tree
5. Repeat until convergence or max iterations

Dataset: Customer Churn Data with features including age, tenure, charges, and contract type.

## Import Required Libraries

In [ ]:
# Data manipulation libraries
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# XGBoost library
import xgboost as xgb

# Scikit-learn utilities for preprocessing and model evaluation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## Step 1: Load and Explore the Dataset

In [ ]:
# Generate synthetic customer churn dataset for a complete runnable example
np.random.seed(42)
n_samples = 1000

age = np.random.randint(18, 70, size=n_samples)
tenure = np.random.randint(1, 72, size=n_samples)
monthly_charges = np.random.uniform(20.0, 120.0, size=n_samples)
total_charges = tenure * monthly_charges * np.random.uniform(0.9, 1.1, size=n_samples)
num_support_calls = np.random.poisson(lam=2, size=n_samples)
contract_type = np.random.choice(['Monthly', 'OneYear', 'TwoYear'], size=n_samples, p=[0.5, 0.3, 0.2])

# Define probability of churn based on features
churn_prob = 1 / (1 + np.exp(-(-2.0 + 0.05 * age - 0.08 * tenure + 0.02 * monthly_charges + 0.3 * num_support_calls + (contract_type == 'Monthly') * 1.5)))
churn = np.random.binomial(1, churn_prob)

# Combine into a pandas DataFrame
data = pd.DataFrame({
    'Age': age,
    'Tenure': tenure,
    'MonthlyCharges': monthly_charges,
    'TotalCharges': total_charges,
    'NumSupport Calls': num_support_calls,
    'ContractType': contract_type,
    'Churn': churn
})

# Display the first few rows to understand the data structure
print("First 5 rows of the generated dataset:")
print(data.head())

# Display basic information about the dataset
print("\nDataset shape:", data.shape)
print("\nColumn names and types:")
print(data.dtypes)

# Check for missing values
print("\nMissing values per column:")
print(data.isnull().sum())

In [ ]:
# Display statistical summary of numerical features
print("Statistical summary of numerical features:")
print(data.describe())

# Check the distribution of the target variable (Churn)
print("\nTarget variable distribution:")
print(data['Churn'].value_counts())
print("\nChurn rate: {:.2f}%".format(data['Churn'].mean() * 100))

## Step 2: Data Preprocessing

In [ ]:
# Initialize LabelEncoder to convert categorical variables to numerical
le = LabelEncoder()

# Encode the ContractType column
# This converts categorical values (Monthly, OneYear, TwoYear) to integers (0, 1, 2)
data['ContractType_Encoded'] = le.fit_transform(data['ContractType'])

# Display the mapping of original values to encoded values
print("Contract Type Encoding:")
for i, label in enumerate(le.classes_):
    print(f"{label} -> {i}")

In [ ]:
# Define the feature columns that will be used for training
feature_columns = [
    'Age',
    'Tenure',
    'MonthlyCharges',
    'TotalCharges',
    'NumSupport Calls',
    'ContractType_Encoded'
]

# Separate features (X) and target variable (y)
# X contains all the input features used for prediction
X = data[feature_columns].values

# y contains the target variable we want to predict (Churn: 0 or 1)
y = data['Churn'].values

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)
print("\nFeature names:", feature_columns)

## Step 3: Split Data into Training and Testing Sets

In [ ]:
# Split the dataset into training and testing sets
# test_size=0.2 means 20% of data will be used for testing
# random_state=42 ensures reproducibility
# stratify=y ensures that the train and test sets have the same proportion of churned customers
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set size:", X_train.shape[0])
print("Testing set size:", X_test.shape[0])
print("\nTraining set churn rate: {:.2f}%".format(y_train.mean() * 100))
print("Testing set churn rate: {:.2f}%".format(y_test.mean() * 100))

## Step 4: Initialize XGBoost Model

Following the XGBoost algorithm flowchart:
- Set the number of boosting rounds (n_estimators)
- Configure the learning rate (eta)
- Set tree-specific parameters (max_depth, min_child_weight)
- Configure regularization parameters (lambda, gamma)

In [ ]:
# Create XGBoost classifier with specified hyperparameters
model = xgb.XGBClassifier(
    # Number of boosting rounds (number of trees to build sequentially)
    n_estimators=100,

    # Learning rate (eta in the flowchart)
    # Controls the contribution of each tree to the final prediction
    # Lower values make the model more robust but require more trees
    learning_rate=0.1,

    # Maximum depth of each tree
    # Deeper trees can model more complex patterns but may overfit
    max_depth=5,

    # Minimum sum of instance weight (hessian) needed in a child node
    # Higher values prevent the model from learning overly specific patterns
    min_child_weight=1,

    # Gamma (minimum loss reduction required to make a split)
    # Larger values make the algorithm more conservative
    gamma=0,

    # Subsample ratio of the training instances
    # Setting it to 0.8 means randomly sample 80% of training data for each tree
    subsample=0.8,

    # Subsample ratio of columns when constructing each tree
    # Setting it to 0.8 means randomly sample 80% of features for each tree
    colsample_bytree=0.8,

    # L1 regularization term on weights (alpha)
    reg_alpha=0,

    # L2 regularization term on weights (lambda in the flowchart)
    reg_lambda=1,

    # Random seed for reproducibility
    random_state=42,

    # Use all available CPU cores for parallel processing
    n_jobs=-1,

    # Evaluation metric used during training
    eval_metric='logloss'
)

print("XGBoost model initialized with the following parameters:")
print(model.get_params())

## Step 5: Train the XGBoost Model

This step implements the main loop from the flowchart:
1. Initialize F_0(x) with initial prediction
2. For m = 1 to n_estimators:
   - Compute gradients (g_i) and hessians (h_i) for each sample
   - Build tree_m to minimize the objective function
   - Update model: F_m(x) = F_(m-1)(x) + eta * tree_m(x)

In [ ]:
# Train the model on the training data
# The fit method performs the iterative boosting process:
# - Starts with an initial prediction (usually log-odds for classification)
# - For each boosting round:
#   - Calculates gradients and hessians of the loss function
#   - Builds a new tree to predict these residuals
#   - Adds the tree to the ensemble with a weight determined by learning_rate

model.fit(
    X_train,
    y_train,
    # Evaluate performance on test set after each boosting round
    eval_set=[(X_train, y_train), (X_test, y_test)],
    # Suppress detailed output during training
    verbose=False
)

print("Model training completed.")
print("Number of trees built:", model.n_estimators)

## Step 6: Analyze Training Progress

Visualize how the model performance improves with each boosting iteration

In [ ]:
# Retrieve the evaluation results stored during training
results = model.evals_result()

# Extract training and validation loss for each boosting round
train_loss = results['validation_0']['logloss']
test_loss = results['validation_1']['logloss']

# Create array of boosting rounds for x-axis
epochs = range(1, len(train_loss) + 1)

# Plot the training and validation loss curves
plt.figure(figsize=(10, 6))
plt.plot(epochs, train_loss, label='Training Loss', linewidth=2)
plt.plot(epochs, test_loss, label='Validation Loss', linewidth=2)
plt.xlabel('Boosting Round')
plt.ylabel('Log Loss')
plt.title('XGBoost Training Progress: Loss vs Boosting Rounds')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Initial training loss: {:.4f}".format(train_loss[0]))
print("Final training loss: {:.4f}".format(train_loss[-1]))
print("Initial validation loss: {:.4f}".format(test_loss[0]))
print("Final validation loss: {:.4f}".format(test_loss[-1]))

## Step 7: Feature Importance Analysis

Examine which features contribute most to the model's predictions

In [ ]:
# Get feature importance scores from the trained model
# These scores indicate how useful each feature was in building the trees
importance_scores = model.feature_importances_

# Create a DataFrame to organize and sort the importance scores
feature_importance_df = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': importance_scores
}).sort_values('Importance', ascending=False)

print("Feature Importance Ranking:")
print(feature_importance_df)

# Visualize feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'])
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.title('Feature Importance in XGBoost Model')
plt.tight_layout()
plt.show()

## Step 8: Generate Predictions

Use the trained model to make predictions on the test set

In [ ]:
# Generate binary predictions (0 or 1) for the test set
# The model predicts 1 (churn) if the probability > 0.5, otherwise 0
y_pred = model.predict(X_test)

# Generate probability predictions for each class
# Returns an array with two columns: [probability of class 0, probability of class 1]
y_pred_proba = model.predict_proba(X_test)

# Display sample predictions
print("Sample predictions (first {} test samples):".format(len(y_test)))
print("\nActual | Predicted | Prob(Stay) | Prob(Churn)")
print("-" * 50)
for i in range(len(y_test)):
    print("{:6d} | {:9d} | {:10.4f} | {:11.4f}".format(
        y_test[i],
        y_pred[i],
        y_pred_proba[i][0],
        y_pred_proba[i][1]
    ))

## Step 9: Evaluate Model Performance

Calculate various metrics to assess the model's predictive accuracy

In [ ]:
# Calculate evaluation metrics

# Accuracy: proportion of correct predictions
accuracy = accuracy_score(y_test, y_pred)

# Precision: of all predicted churns, how many actually churned
precision = precision_score(y_test, y_pred)

# Recall: of all actual churns, how many were correctly predicted
recall = recall_score(y_test, y_pred)

# F1-score: harmonic mean of precision and recall
f1 = f1_score(y_test, y_pred)

# AUC-ROC: area under the receiver operating characteristic curve
# Measures the model's ability to distinguish between classes
auc_roc = roc_auc_score(y_test, y_pred_proba[:, 1])

# Print all metrics
print("Model Performance Metrics:")
print("=" * 40)
print("Accuracy:  {:.4f}".format(accuracy))
print("Precision: {:.4f}".format(precision))
print("Recall:    {:.4f}".format(recall))
print("F1-Score:  {:.4f}".format(f1))
print("AUC-ROC:   {:.4f}".format(auc_roc))

In [ ]:
# Generate and display the confusion matrix
# Shows the distribution of true positives, true negatives, false positives, and false negatives
cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)

# Extract values from confusion matrix
tn, fp, fn, tp = cm.ravel()

print("\nBreakdown:")
print("True Negatives (correctly predicted no churn): {}".format(tn))
print("False Positives (incorrectly predicted churn): {}".format(fp))
print("False Negatives (incorrectly predicted no churn): {}".format(fn))
print("True Positives (correctly predicted churn): {}".format(tp))

# Visualize confusion matrix as a heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Generate and display the classification report
# Provides precision, recall, and f1-score for each class
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

## Step 10: Plot ROC Curve

Visualize the trade-off between true positive rate and false positive rate

In [ ]:
# Calculate ROC curve points
# fpr: false positive rate at different thresholds
# tpr: true positive rate at different thresholds
# thresholds: probability thresholds used to compute fpr and tpr
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba[:, 1])

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, linewidth=2, label='XGBoost (AUC = {:.3f})'.format(auc_roc))
# Plot diagonal line representing random guessing
plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 11: Make Predictions on New Data

Apply the trained model to predict churn for new customers

In [ ]:
# Define new customer data for prediction
# Each row represents a customer with the following features:
# [Age, Tenure, MonthlyCharges, TotalCharges, NumSupportCalls, ContractType_Encoded]

new_customers = np.array([
    [28, 3, 110.0, 330, 8, 0],    # Young customer, short tenure, high charges, many support calls, monthly contract
    [58, 48, 55.3, 2654, 0, 2],   # Older customer, long tenure, low charges, no support calls, two-year contract
    [45, 12, 85.5, 1026, 5, 1]    # Middle-aged customer, moderate tenure, moderate charges, some support calls, one-year contract
])

# Generate predictions for new customers
new_predictions = model.predict(new_customers)
new_probabilities = model.predict_proba(new_customers)

# Display predictions for each new customer
print("Predictions for New Customers:")
print("=" * 70)
for i in range(len(new_customers)):
    print("\nCustomer {}:".format(i + 1))
    print("  Features: Age={}, Tenure={}, Monthly=${:.1f}, Total=${:.0f}, Calls={}, Contract={}".format(
        new_customers[i][0],
        new_customers[i][1],
        new_customers[i][2],
        new_customers[i][3],
        new_customers[i][4],
        new_customers[i][5]
    ))
    print("  Prediction: {}".format('Churn' if new_predictions[i] == 1 else 'No Churn'))
    print("  Probability of Churn: {:.2%}".format(new_probabilities[i][1]))
    print("  Probability of No Churn: {:.2%}".format(new_probabilities[i][0]))

## Step 12: Model Summary

Display key information about the trained model

In [ ]:
# Print comprehensive model summary
print("XGBoost Model Summary")
print("=" * 50)
print("\nModel Configuration:")
print("  Number of boosting rounds: {}".format(model.n_estimators))
print("  Learning rate: {}".format(model.learning_rate))
print("  Maximum tree depth: {}".format(model.max_depth))
print("  Minimum child weight: {}".format(model.min_child_weight))
print("  Gamma: {}".format(model.gamma))
print("  Subsample ratio: {}".format(model.subsample))
print("  Column subsample ratio: {}".format(model.colsample_bytree))
print("  L1 regularization (alpha): {}".format(model.reg_alpha))
print("  L2 regularization (lambda): {}".format(model.reg_lambda))

print("\nTraining Data:")
print("  Training samples: {}".format(X_train.shape[0]))
print("  Testing samples: {}".format(X_test.shape[0]))
print("  Number of features: {}".format(X_train.shape[1]))

print("\nModel Performance:")
print("  Accuracy: {:.2%}".format(accuracy))
print("  Precision: {:.2%}".format(precision))
print("  Recall: {:.2%}".format(recall))
print("  F1-Score: {:.2%}".format(f1))
print("  AUC-ROC: {:.2%}".format(auc_roc))

print("\nTop 3 Most Important Features:")
for i in range(min(3, len(feature_importance_df))):
    row = feature_importance_df.iloc[i]
    print("  {}: {:.4f}".format(row['Feature'], row['Importance']))